In [1]:
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.v2 as T
from IPython.display import clear_output
from PIL import Image
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.optim import Optimizer

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
import torch
from torchmetrics.classification import BinaryF1Score


def find_best_threshold(probs: torch.Tensor, targets: torch.Tensor, device: torch.device):
    best_threshold = 0.5
    best_f1 = 0.0

    thresholds = torch.arange(0.1, 0.91, 0.05)

    for threshold in thresholds:
        metric = BinaryF1Score(threshold=float(threshold)).to(device)
        f1 = metric(probs, targets).item()

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = float(threshold)

    return best_threshold, best_f1

In [3]:
from torchmetrics.classification import BinaryF1Score

def train(model: nn.Module, data_loader: DataLoader, optimizer: Optimizer, loss_fn, device: torch.device):
    model.train()

    total_loss = 0.0

    for x, y in tqdm(data_loader):
        x = x.to(device)
        y = y.to(device).float().unsqueeze(1)

        optimizer.zero_grad()

        output = model(x)
        loss = loss_fn(output, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


In [3]:
@torch.inference_mode()
def evaluate(model: nn.Module, data_loader: DataLoader, loss_fn, device: torch.device):
    model.eval()

    total_loss = 0.0
    all_probs = []
    all_targets = []

    for x, y in tqdm(data_loader):
        x = x.to(device)
        y = y.to(device).float().unsqueeze(1)

        output = model(x)
        loss = loss_fn(output, y)
        total_loss += loss.item()

        probs = torch.sigmoid(output).squeeze(1)   
        targets = y.squeeze(1).int()              
        all_probs.append(probs)
        all_targets.append(targets)

    all_probs = torch.cat(all_probs)
    all_targets = torch.cat(all_targets)

    best_threshold, best_f1 = find_best_threshold(all_probs, all_targets, device)

    return total_loss / len(data_loader), best_f1, best_threshold

In [4]:
def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    valid_f1: list[float],
    title: str
):
    plt.figure(figsize=(16, 8))

    plt.title(title + ' loss')

    plt.plot(train_loss, label='Train loss')
    plt.plot(valid_loss, label='Valid loss')
    plt.legend()

    plt.show()

    plt.figure(figsize=(16, 8))

    plt.title(title + ' f1')

    plt.plot(valid_f1, label='Valid f1')
    plt.legend()

    plt.show()

In [5]:
def fit(model, train_loader, valid_loader, optimizer, loss_fn, device, num_epochs, title):
    train_loss_history, valid_loss_history = [], []
    valid_f1_history = []
    threshold_history = []

    best_valid_f1 = -1.0
    best_threshold = 0.5

    for epoch in range(num_epochs):
        train_loss = train(model, train_loader, optimizer, loss_fn, device)
        valid_loss, valid_f1, valid_threshold = evaluate(model, valid_loader, loss_fn, device)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)
        valid_f1_history.append(valid_f1)
        threshold_history.append(valid_threshold)

        clear_output()

        plot_stats(
            train_loss_history,
            valid_loss_history,
            valid_f1_history,
            title
        )

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"valid_loss={valid_loss:.4f} | "
            f"valid_f1={valid_f1:.4f} | "
            f"best_thr={valid_threshold:.2f}"
        )

        if valid_f1 > best_valid_f1:
            best_valid_f1 = valid_f1
            best_threshold = valid_threshold

            torch.save(model.state_dict(), "best_model.pt")
            torch.save(optimizer.state_dict(), "best_optimizer.pt")
            torch.save({"threshold": best_threshold}, "best_threshold.pt")

            print(f"best model epoch {epoch}")

    print(f"Best valid F1: {best_valid_f1:.4f}")
    print(f"Best threshold: {best_threshold:.2f}")

In [12]:
@torch.inference_mode()
def predict_test(model, test_loader, device, threshold=0.5):
    model.eval()

    all_ids = []
    all_preds = []

    for x, ids in tqdm(test_loader):
        x = x.to(device)

        output = model(x)
        res = torch.sigmoid(output).squeeze(1)
        preds = (res > threshold).long()

        all_ids.extend(ids.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

    submission = pd.DataFrame({
        "id": all_ids,
        "target_feature": all_preds
    })

    submission = submission.sort_values("id").reset_index(drop=True)
    submission.to_csv(f"{model}.csv", index=False)

    return submission

In [8]:
import os
from PIL import Image
from torch.utils.data import DataLoader

def make_test_image_loader(
    folder_path,
    transform=None,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    return_ids=True,
):
    image_files = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith(".jpg")],
        key=lambda x: int(os.path.splitext(x)[0])
    )

    data = []

    for filename in image_files:
        path = os.path.join(folder_path, filename)
        image = np.array(Image.open(path).convert("RGB"))

        if transform is not None:
            image = transform(image=image)["image"]

        if return_ids:
            img_id = int(os.path.splitext(filename)[0])
            data.append((image, img_id))
        else:
            data.append(image)

    return DataLoader(
        data,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True
    )

In [9]:
from sklearn.model_selection import train_test_split


def make_train_valid_loaders(
    images_folder,
    labels_csv,
    train_transform=None,
    valid_transform=None,
    batch_size=32,
    valid_size=0.2,
    random_state=42,
    shuffle_train=True,
    num_workers=0
):
    df = pd.read_csv(labels_csv, header=None, names=["Id", "target_feature"])

    train_df, valid_df = train_test_split(
        df,
        test_size=valid_size,
        random_state=random_state,
        stratify=df["target_feature"]
    )

    train_items = [
        (os.path.join(images_folder, f"{int(row.Id)}.jpg"), int(row.target_feature))
        for row in train_df.itertuples(index=False)
    ]
    valid_items = [
        (os.path.join(images_folder, f"{int(row.Id)}.jpg"), int(row.target_feature))
        for row in valid_df.itertuples(index=False)
    ]

    def make_collate(transform):
        def collate_fn(batch):
            images = []
            labels = []

            for path, label in batch:
                image = np.array(Image.open(path).convert("RGB"))
                if transform is not None:
                    image = transform(image=image)["image"]
                images.append(image)
                labels.append(label)

            images = torch.stack(images)
            labels = torch.tensor(labels, dtype=torch.long)
            return images, labels
        return collate_fn

    train_loader = DataLoader(
        train_items,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        collate_fn=make_collate(train_transform),
        pin_memory=True
    )

    valid_loader = DataLoader(
        valid_items,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=make_collate(valid_transform),
        pin_memory=True
    )

    return train_loader, valid_loader

In [9]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.MedianBlur(blur_limit=(3, 3), p=0.2),
    A.Normalize(mean=(0.5192, 0.4276, 0.3844),
                std=(0.2587, 0.2372, 0.2334)),
    ToTensorV2(),
])

test_transforms = A.Compose([
    A.Normalize(mean=(0.5192, 0.4276, 0.3844),
                std=(0.2587, 0.2372, 0.2334)),
    ToTensorV2(),
])

In [ ]:
train_loader, valid_loader = make_train_valid_loaders(images_folder="/kaggle/input/datasets/hawwkps/dataset/dataset/train_images",labels_csv="/kaggle/input/datasets/hawwkps/dataset/dataset/train_solution.csv",train_transform=train_transforms,valid_transform=test_transforms,batch_size=64,valid_size=0.2,random_state=100,shuffle_train=True, num_workers=4)
test_loader = make_test_image_loader('/kaggle/input/datasets/hawwkps/dataset/dataset/test_images',transform=test_transforms, batch_size=64, shuffle= False,  num_workers=4)

In [13]:
df = pd.read_csv("/kaggle/input/datasets/hawwkps/dataset/dataset/train_solution.csv", header=None, names=["Id", "target_feature"])

num_pos = (df["target_feature"] == 1).sum()
num_neg = (df["target_feature"] == 0).sum()

pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

In [ ]:
import torch
import torch.nn as nn


class SeBlock(nn.Module):
    def __init__(self, in_channels, reduction = 16):
        super().__init__()
        hidden = max(in_channels // reduction, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        self.block = nn.Sequential(
            nn.Linear(in_channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, in_channels),
            nn.Sigmoid(),
        )

    def forward(self,x):
        b,c, _,_ = x.shape

        w = self.pool(x)
        w = w.view(b,c)
        w = self.block(w)
        w = w.view(b,c,1,1)

        return x * w


class Block(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,out_channels=out_channels,kernel_size=3,stride=stride,padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),

            nn.Conv2d(in_channels=out_channels,out_channels=out_channels,kernel_size=3,padding=1),
            nn.BatchNorm2d(out_channels),
            SeBlock(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv_block(x)


class ResNet_SE(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.block1 = Block(in_channels, 32, stride=2)
        self.block2 = Block(32, 64)

        self.block3 = Block(32 + 64, 128, stride=2)
        self.block4 = Block(128, 128)


        self.block5 = Block(128 + 128, 256, stride=2)
        self.block6 = Block(256, 256)

        
        self.block7 = Block(256 + 256, 512, stride=2)
        self.block8 = Block(512, 512)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 + 512, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x1 = self.block1(x)                
        x2 = self.block2(x1)               
        x2_cat = torch.cat([x1, x2], dim=1)  

        x3 = self.block3(x2_cat)           
        x4 = self.block4(x3)               
        x4_cat = torch.cat([x3, x4], dim=1)  

        x5 = self.block5(x4_cat)           
        x6 = self.block6(x5)               
        x6_cat = torch.cat([x5, x6], dim=1)  


        x7 = self.block7(x6_cat)          
        x8 = self.block8(x7)              
        x8_cat = torch.cat([x7, x8], dim=1)  

        p = self.pool(x8_cat)              
        out = self.classifier(p)           

        return out

In [ ]:
ResNetSE = ResNet_SE(3).to(device)
ResNetSE_loss_fn = nn.BCEWithLogitsLoss(pos_weight = pos_weight).to(device)
ResNetSE_optimizer = torch.optim.Adam(ResNetSE.parameters(), lr=3e-4, weight_decay = 1e-4)

In [ ]:
fit(ResNetSE, train_loader, valid_loader, ResNetSE_optimizer, ResNetSE_loss_fn, device, num_epochs=70, title="ResNetSE")

In [ ]:
torch.save(ResNetSE.state_dict(), "ResNetSE.pt")

In [ ]:
ResNetSE_submission = predict_test(ResNetSE, test_loader, device, threshold = 0.93)